Possible obfuscators

https://pyobfuscate.com/
https://github.com/davidteather/python-obfuscator
https://pyob.oxyry.com/

In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import python_obfuscator
from networkx import DiGraph
from python_obfuscator.techniques import (
    add_random_variables,
    one_liner,
    variable_renamer,
)
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from source.python2json import PythonCodeParser

In [41]:
def read_file_content(file_path: Path) -> str:
    with file_path.open("r") as file:
        return file.read()


def code_to_json(source_code: str) -> dict:
    return PythonCodeParser(source_code).traverse_tree()


def file_to_code_to_json(file_path: str) -> dict:
    return code_to_json(read_file_content(Path(file_path)))

# Building dataset

In [5]:
binary_insertion_sort = """def binary_search(arr, key, low, high):
    if low >= high:
        return low + 1 if key > arr[low] else low
    mid = low + (high - low) // 2
    if arr[mid] == key:
        return mid + 1
    elif arr[mid] > key:
        return binary_search(arr, key, low, mid - 1)
    else:
        return binary_search(arr, key, mid + 1, high)

def insertion_sort(arr, size):
    for i in range(1, size):
        j = i - 1
        key = arr[i]
        index = binary_search(arr, key, 0, j)
        while j >= index:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key

if __name__ == "__main__":
    n = int(input("Enter size of array:\n"))
    arr = [int(input("Enter the elements of the array\n")) for _ in range(n)]
    print("Original array:", arr)
    insertion_sort(arr, n)
    print("Sorted array:", arr)
"""

binary_insertion_sort_oxyry = """def binary_search (O0O000OOO000OOOOO ,O0000000OOOOO0OOO ,O0OOO00O000O0OOO0 ,O00O0OOO000OOOO0O ):#line:1
    if O0OOO00O000O0OOO0 >=O00O0OOO000OOOO0O :#line:2
        return O0OOO00O000O0OOO0 +1 if O0000000OOOOO0OOO >O0O000OOO000OOOOO [O0OOO00O000O0OOO0 ]else O0OOO00O000O0OOO0 #line:3
    O0O0O0OO0O0OO0OO0 =O0OOO00O000O0OOO0 +(O00O0OOO000OOOO0O -O0OOO00O000O0OOO0 )//2 #line:4
    if O0O000OOO000OOOOO [O0O0O0OO0O0OO0OO0 ]==O0000000OOOOO0OOO :#line:5
        return O0O0O0OO0O0OO0OO0 +1 #line:6
    elif O0O000OOO000OOOOO [O0O0O0OO0O0OO0OO0 ]>O0000000OOOOO0OOO :#line:7
        return binary_search (O0O000OOO000OOOOO ,O0000000OOOOO0OOO ,O0OOO00O000O0OOO0 ,O0O0O0OO0O0OO0OO0 -1 )#line:8
    else :#line:9
        return binary_search (O0O000OOO000OOOOO ,O0000000OOOOO0OOO ,O0O0O0OO0O0OO0OO0 +1 ,O00O0OOO000OOOO0O )#line:10
def insertion_sort (O0O00000OOO0000O0 ,O000OOO0OOOOOO0O0 ):#line:12
    for O0OO0OOO00OOOO000 in range (1 ,O000OOO0OOOOOO0O0 ):#line:13
        OO000OO000O00OO00 =O0OO0OOO00OOOO000 -1 #line:14
        OOOO0O0O00O0OOO00 =O0O00000OOO0000O0 [O0OO0OOO00OOOO000 ]#line:15
        O0OOO0OO000OO0O00 =binary_search (O0O00000OOO0000O0 ,OOOO0O0O00O0OOO00 ,0 ,OO000OO000O00OO00 )#line:16
        while OO000OO000O00OO00 >=O0OOO0OO000OO0O00 :#line:17
            O0O00000OOO0000O0 [OO000OO000O00OO00 +1 ]=O0O00000OOO0000O0 [OO000OO000O00OO00 ]#line:18
            OO000OO000O00OO00 -=1 #line:19
        O0O00000OOO0000O0 [OO000OO000O00OO00 +1 ]=OOOO0O0O00O0OOO00 #line:20
if __name__ =="__main__":#line:22
    n =int (input ("Enter size of array:\n"))#line:23
    arr =[int (input ("Enter the elements of the array\n"))for _O0000O0OO00OOO0OO in range (n )]#line:24
    print ("Original array:",arr )#line:25
    insertion_sort (arr ,n )#line:26
    print ("Sorted array:",arr )#line:27
"""


In [4]:
def obfuscate(program: str, strategy: int):
    """
    Obfuscates a Python program.

    - program: the program to obfuscate, represented as a string
    - strategy: the obfuscation strategy to use (integer number from 0 to 7)
    """
    obfuscation_techniques = [
        [],  # 0 0 0
        [variable_renamer],
        [one_liner],
        [one_liner, variable_renamer],
        [add_random_variables],
        [add_random_variables, variable_renamer],
        [add_random_variables, one_liner],
        # [add_random_variables, one_liner, variable_renamer] | This won't do any obfuscation at all, and thus will be discarded
    ]

    return python_obfuscator.obfuscator().obfuscate(
        program, remove_techniques=obfuscation_techniques[strategy]
    )


# Generating all obfuscations for a given problem
# The strategies that passed the test in experiments.ipynb are deeemed safe,
# but we'll still have to double check the results anyway
SAFE_STRATEGIES = [2, 3, 6]


def generate_obfuscations(program: str):
    return [obfuscate(program, strategy) for strategy in SAFE_STRATEGIES]

In [ ]:
generate_obfuscations(binary_insertion_sort)

# idea 1: Use networkx to compare the graphs using the GED algorithm

Using the networkx to build the professor's idea

In [42]:
print1 = """print("HELLO")
"""

print2 = """print("hello")
"""

In [43]:
def json_to_nx(data: dict) -> DiGraph:
    G = nx.DiGraph()

    for key, value in data.items():
        if key in ["language", "ignored"]:
            continue
        for id, item in enumerate(value):
            if item.get("name"):
                continue
            node_id = f"{key}-{id}"
            G.add_node(
                node_id,
                content=item.get("content"),
                row=item.get("row"),
                column=item.get("column"),
            )
            for edge in item.get("edge", []):
                G.add_edge(node_id, edge)
    return G


def file_to_code_to_json_to_nx(file_path: Path) -> DiGraph:
    return json_to_nx(code_to_json(read_file_content(file_path)))


def code_to_json_to_nx(source_code: str) -> DiGraph:
    return json_to_nx(code_to_json(source_code))

In [44]:
G1 = code_to_json_to_nx(print1)
G2 = code_to_json_to_nx(print2)

In [ ]:
# Visualization with the 'name' attribute as labels
node_labels = nx.get_node_attributes(G1, "content")  # Get 'name' attribute for labeling
pos = nx.spring_layout(G1)  # Layout for visualization
nx.draw(
    G1,
    pos,
    with_labels=True,
    labels=node_labels,
    node_size=3000,
    node_color="lightblue",
)
plt.show()

After loading into networkx, test simple isomorphism of networkx

In [ ]:
from networkx.algorithms import isomorphism

GM = isomorphism.DiGraphMatcher(G1, G2)
is_iso = GM.is_isomorphic()
is_iso

In [ ]:
from networkx.algorithms import graph_edit_distance

graph_edit_distance(G1, G2)

Here you can use custom isomorphisms, interesting idea

In [ ]:
def node_match(n1, n2):
    return n1["content"] == n2["content"]


GM = isomorphism.DiGraphMatcher(G1, G2, node_match=node_match)
GM.is_isomorphic()

The result is that it needs to create a way to match the node contents.

Now I want to use the [GED algorithm](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.similarity.graph_edit_distance.html#networkx.algorithms.similarity.graph_edit_distance) to test 13A python only

In [152]:
code1 = """from math import log,ceil,gcd

n = int(input())
a = 0
for i in range(2,n):
    m = n
    while m:
        m,r = divmod(m,i)
        a += r

g = gcd(a,n-2)
print(f"{a//g}/{(n-2)//g}")"""
code2 = """from math import gcd

def sanoq(n,x):
    s=0
    while n:
        s, n = s+n%x, n//x
    return s

a=int(input())
j=0
for x in range(2,a):
    j+=sanoq(a,x)

g=gcd(j,a-2)
print(f"{j//g}/{(a-2)//g}")"""
code3 = """import math

a=int(input())
r=0
for b in range(2,a):
  c=a
  while c:r+=c%b;c//=b

a-=2
d=math.gcd(r,a)
print(f'{r//d}/{a//d}')"""
code4 = """import sys

input = sys.stdin.readline
A = int(input())
res = 0
def calc(x, base):
    v = 0
    while x > 0:
        v += x % base
        x //= base
    return v



def gcd(a, b):
    return a if b == 0 else gcd(b, a%b)



for i in range(2, A):
    res += calc(A, i)



d = gcd(res, A-2)
print(f'{res//d}/{(A-2)//d}')
"""
code5 = """import math


a = int(input())
s = 0
for i in range(2, a):
    tmpA = a
    while tmpA != 0:
        s = s + (tmpA % i)
        tmpA = (tmpA // i)
d = math.gcd(s, a-2)
print(f"{s//d}/{(a-2)//d}")
"""

code6 = """for x in range(int(input())) : 

    n,k=map(int,input().split())

    a=input()

    arr=list(map(int,a.split()))

    arr1=(list(map(int,a.split())))

    d={}

    arr1.sort()

    i,c=0,0

    for j in range(n) :  d[arr1[j]]=j #d.update({arr1[j], j})

    for i in range(n) : 

        if d[arr[i]]>0 and i>0 and arr1[d[arr[i]]-1]==arr[i-1] :continue 

        else :c+=1

    if c<=k : print("Yes")

    else:print("No") 

"""
codes = [code1, code2, code3, code4, code5]
codes1 = [code1, code6]

In [ ]:
graphs = []
for idx, code in enumerate(codes):
    graphs.append((code_to_json_to_nx(code), idx))

for idx, tuplax in enumerate(graphs):
    for idy in range(idx + 1, len(graphs)):
        tuplay = graphs[idy]
        print(
            f"({tuplax[1]+1}-{tuplay[1]+1}): {graph_edit_distance(tuplax[0], tuplay[0])}"
        )

the behavior here is good, the interpretability of this is the structure, when the ged is equal to zero, it means the structure of the graph is identical.

My hypothesis here is that the bigger the GED the more different the code is.

The problem is the large the codebase and comparisons, more time it takes, on my computer it took 31s to compare 5 small codes from `codeforces A 13` contest.


# Idea 2: Using TFIDFVectorizer

## 2.1 using the blocks from X9 as documents to test the tfidf idea

In [ ]:
# pick only the content form dict
def create_documents(data: dict) -> list[str]:
    resp = []
    for key, value in data.items():
        if key in ["language", "ignored"]:
            continue
        for id, item in enumerate(value):
            if item.get("name"):
                continue
            resp.append(item.get("content"))
    return resp


def pick_all_documents(codes: list[str]) -> list[str]:
    corpus = []
    for file_path in codes:
        corpus += create_documents(code_to_json(file_path))
    return corpus


corpus = pick_all_documents(codes)
corpus

In [69]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

In [ ]:
def generate_graphs_from_codes(codes: list[str]) -> list[DiGraph]:
    # load, create corpus and node_info_list
    corpus = []
    node_info_list = []
    for file_path in codes:
        data = code_to_json(file_path)
        node_info = []
        for key, value in data.items():
            if key in ["language", "ignored"]:
                continue
            for item in value:
                if item.get("name"):
                    continue
                content = item.get("content", "")
                corpus.append(content)
                node_info.append(
                    {
                        "key": key,
                        "row": item.get("row"),
                        "column": item.get("column"),
                        "edge": item.get("edge"),
                        "content": content,
                    }
                )
        node_info_list.append(node_info)

    # Compute TF-IDF vectors
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus)
    tfidf_vectors = tfidf_matrix.toarray()

    # generate the graphs
    graphs = []
    tfidf_index = 0
    for node_info in node_info_list:
        G = nx.DiGraph()
        for id, node in enumerate(node_info, start=1):
            node_id = f"{node['key']}-{id}"
            tfidf_vector = tfidf_vectors[tfidf_index]
            tfidf_index += 1
            G.add_node(node_id, tfidf=tfidf_vector, **node)
            for edge in node["edge"]:
                G.add_edge(node_id, v_of_edge=edge)
        graphs.append(G)
    return graphs


graphs = generate_graphs_from_codes(codes)
nx.draw_networkx(graphs[1])
plt.show()

In [ ]:
nx.draw_networkx(graphs[0])
plt.show()

With this structure, how can I compare the graphs?


## Maybe the tfidf only compares what documents are more peculiar than others?

In [ ]:
# Sample documents
documents = [
    "I love programming in Python.",
    "Python programming is fun.",
    "I love coding."
]

# Initialize the vectorizer with custom parameters
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

# Fit and transform the documents
tfidf_matrix = vectorizer.fit_transform(documents)

# Get feature names
feature_names = vectorizer.get_feature_names_out()

# Convert TF-IDF matrix to array
tfidf_array = tfidf_matrix.toarray()

# Display the TF-IDF scores
df = pd.DataFrame(tfidf_array, columns=feature_names)
print(df)

# display similarities
similarity = cosine_similarity(tfidf_matrix)
similar_pairs = []
threshold = 0.2
for i in range(len(similarity)):
    for j in range(i + 1, len(similarity)):
        if similarity[i, j] >= threshold:
            similar_pairs.append((documents[i], documents[j], similarity[i, j]))
print(similar_pairs)


Creating a function that genereates the similares and show the codes if a defined threshold

In [145]:
def gen_and_show_similarities(documents: list[str], threshold: float = 0.8) -> None:
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)

    similarity = cosine_similarity(tfidf_matrix)
    similar_pairs = []
    for i in range(len(similarity)):
        for j in range(i + 1, len(similarity)):
            if similarity[i, j] >= threshold:
                similar_pairs.append(
                    [documents[i], documents[j], similarity[i, j], (i, j)]
                )

    for i in similar_pairs:
        print(f"====================== Similarity between {i[3]}: {i[2]:.2f}")
        print(f"====================== START CODE{i[3][0]} ======================")
        print(i[0])
        print(f"====================== END CODE{i[3][0]} ======================")
        print()
        print(f"====================== START CODE{i[3][1]} ======================")
        print(i[1])
        print(f"====================== END CODE{i[3][1]} ======================")
        print("\n\n\n")

In [ ]:
gen_and_show_similarities(codes,threshold=0.4)

Testando com o corpus, ou seja separado em grafos

In [ ]:
gen_and_show_similarities(corpus,threshold=0.9)

use the json as str and test it

In [ ]:
codes_strdict = [json.dumps(code_to_json(code)) for code in codes]
gen_and_show_similarities(codes_strdict)

Is the similarity high because of the text is trash?

# Idea: use the kmeans

In [ ]:
# Sample code snippets as documents
documents = [
    "def add(a, b): return a + b",
    "int add(int a, int b) { return a + b; }",
    "def multiply(a, b): return a * b",
    "int multiply(int a, int b) { return a * b; }",
    # More code snippets...
]

# Initialize the TF-IDF Vectorizer
vectorizer = TfidfVectorizer(token_pattern=r'\b\w+\b')

# Transform documents into TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()
tfidf_array = tfidf_matrix.toarray()
df = pd.DataFrame(tfidf_array, columns=feature_names)
print(df)
# Perform clustering
num_clusters = 3
km = KMeans(n_clusters=num_clusters)
km.fit(tfidf_matrix)

# Output cluster assignments
clusters = km.labels_.tolist()
for i, cluster in enumerate(clusters):
    print(f"Document {i} is in cluster {cluster}")


In [ ]:
corpus = pick_all_documents(to_be_compared_list)
vectorizer = TfidfVectorizer(token_pattern=r'\b\w+\b')
tfidf_matrix = vectorizer.fit_transform(corpus)

feature_names = vectorizer.get_feature_names_out()
tfidf_array = tfidf_matrix.toarray()
df = pd.DataFrame(tfidf_array, columns=feature_names)
print(df)

cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
print(f"Cosine Similarity: {cosine_sim[0][0]:.4f}")

1. ahsuehaseuhsa = 2
1. a= 2
1. b= 2

In [ ]:
corpus

In [ ]:
# Compute cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix)

# Display the cosine similarity matrix
print("Cosine Similarity Matrix:")
for i, row in enumerate(cosine_sim):
    print(f"Document {i} similarities: {row}")


In [ ]:
print(f"Cosine Similarity: {cosine_sim[0][0]:.4f}")

## Investigating and gaining insight on the DOLOS approach

In [ ]:
import tree_sitter_python
from tree_sitter import Language, Parser

PY_LANGUAGE = Language(tree_sitter_python.language())

parser = Parser(PY_LANGUAGE)

code = """
def add(a, b):
    return a + b
"""

tree = parser.parse(bytes(code, "utf8"))
root_node = tree.root_node
root_node

In [ ]:
def traverse_tree(node):
    """Generator to traverse the AST in a depth-first manner."""
    yield node
    for child in node.children:
        yield from traverse_tree(child)


In [ ]:
all_nodes = list(traverse_tree(root_node))
all_nodes

In [ ]:
def generate_kgrams(elements, k):
    """Generate k-grams from a list of elements."""
    return [elements[i:i + k] for i in range(len(elements) - k + 1)]
k = 3
k_grams = generate_kgrams(all_nodes, k)
k_grams
for gram in k_grams:
    print(gram)

In [ ]:
def extract_kgrams(tree, k):
    root_node = tree.root_node
    all_nodes = list(traverse_tree(root_node))
    node_texts = [node.type for node in all_nodes]
    return generate_kgrams(node_texts, k)

In [ ]:
from collections import Counter

code1 = """
def add(a, b):
    return a + b
"""

code2 = """
def sum_numbers(x:int, y:int)->int:
    return a + b
"""

tree1 = parser.parse(bytes(code1, "utf8"))
tree2 = parser.parse(bytes(code2, "utf8"))

k = 5
kgrams1 = extract_kgrams(tree1, k)
kgrams2 = extract_kgrams(tree2, k)


# Flatten k-grams and count frequencies
vector1 = Counter(tuple(gram) for gram in kgrams1)
vector2 = Counter(tuple(gram) for gram in kgrams2)
vector1, vector2

In [ ]:
from math import sqrt


def cosine_similarity(vec1, vec2):
    all_keys = set(vec1.keys()) | set(vec2.keys())
    dot_product = sum(vec1.get(k, 0) * vec2.get(k, 0) for k in all_keys)
    magnitude1 = sqrt(sum(v ** 2 for v in vec1.values()))
    magnitude2 = sqrt(sum(v ** 2 for v in vec2.values()))
    return dot_product / (magnitude1 * magnitude2)

cosine_score = cosine_similarity(vector1, vector2)
print(f"Cosine Similarity: {cosine_score}")

In [ ]:
def get_vectors_kgrams(code1, code2, k=5) -> tuple[Counter, Counter]:
    tree1 = parser.parse(bytes(code1, "utf8"))
    tree2 = parser.parse(bytes(code2, "utf8"))

    kgrams1 = extract_kgrams(tree1, k)
    kgrams2 = extract_kgrams(tree2, k)

    # Flatten k-grams and count frequencies
    vector1 = Counter(tuple(gram) for gram in kgrams1)
    vector2 = Counter(tuple(gram) for gram in kgrams2)
    return vector1, vector2

In [ ]:
vector1, vector2 = get_vectors_kgrams(print1,print2)
cosine_similarity(vector1, vector2)

  for (int x = 0; x < 3; x++) {
        for (int y = 0; y < 3; y++) {
            grid[x][y][0] = x; // Armazena o valor de x
            grid[x][y][1] = y; // Armazena o valor de y
        }
    }

function sum(a, b) {
  return a + b;
}

program ([0, 0] - [3, 0])
function ([0, 0] - [2, 1])
identifier ([0, 9] - [0, 12])
formal_parameters ([0, 12] - [0, 18])
identifier ([0, 13] - [0, 14])
identifier ([0, 16] - [0, 17])
statement_block ([0, 19] - [2, 1])
return_statement ([1, 2] - [1, 15])
binary_expression ([1, 9] - [1, 14])
identifier ([1, 9] - [1, 10])
identifier ([1, 13] - [1, 14])

# idea: Looking into word2vec


In [ ]:
snippet1 = ["def", "add", "(", "a", ",", "b", ")", ":", "return", "a", "+", "b"]
snippet2 = ["def", "sum_list", "(", "lst", ")", ":", "return", "sum", "(", "lst", ")"]
snippet3 = ["def", "total", "(", "arr", ")", ":", "return", "sum", "(", "arr", ")"]
snippet4 = ["def", "multiply", "(", "x", ",", "y", ")", ":", "return", "x", "*","y"]
snippet5 = ["def", "add", "(", "c", ",", "d", ")", ":", "return", "c", "+", "d"]


In [ ]:
from gensim.models import Word2Vec

# Example corpus: tokenized code snippets
code_corpus = [
    snippet1,
    snippet2,
    snippet3,
    snippet4,
    snippet5,
]

# Train Word2Vec
model = Word2Vec(
    sentences=code_corpus, vector_size=50, window=5, min_count=1, workers=4
)

# Save the model for later use
model.save("code_word2vec.model")


In [ ]:
# Load the trained model
model = Word2Vec.load("code_word2vec.model")

# Get vector for the token "return"
vector = model.wv["return"]
print("Vector for 'return':", vector)


In [ ]:
# Tokens most similar to "sum"
similar_tokens = model.wv.most_similar("sum", topn=3)
print("Tokens similar to 'sum':", similar_tokens)

In [ ]:


def average_vector(tokens, model):
    """Get the average vector for a list of tokens."""
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(model.vector_size)


# Compute snippet embeddings
vector1 = average_vector(snippet1, model)
vector2 = average_vector(snippet2, model)
vector3 = average_vector(snippet3, model)
# Cosine similarity
cosine_similarity = np.dot(vector1, vector2) / (
    np.linalg.norm(vector1) * np.linalg.norm(vector2)
)
print("Similarity between snippets:", cosine_similarity)


In [ ]:
def similarity_between_vectors(vector1, vector2):
    # Compute snippet embeddings

    # Cosine similarity
    cosine_similarity = np.dot(vector1, vector2) / (
        np.linalg.norm(vector1) * np.linalg.norm(vector2)
    )
    print("Similarity between snippets:", cosine_similarity)


In [ ]:
similarity_between_vectors(vector1, vector3)

# Idea: Pagerank?

In [ ]:
# Example: Representing a call graph
G = nx.DiGraph()
G.add_edges_from([
    ("main", "function1"),
    ("main", "function2"),
    ("function1", "function3"),
    ("function2", "function3"),
    ("function3", "utility")
])

# Compute PageRank
pagerank_scores = nx.pagerank(G, alpha=0.85)
print("PageRank Scores:", pagerank_scores)


In [ ]:
from scipy.spatial.distance import cosine

# Example PageRank scores for two graphs
pagerank_1 = [0.4, 0.3, 0.2, 0.1]  # Normalized scores from Graph 1
pagerank_2 = [0.35, 0.32, 0.2, 0.13]  # Normalized scores from Graph 2

similarity = 1 - cosine(pagerank_1, pagerank_2)  # Cosine similarity
print("Similarity Score:", similarity)


In [ ]:
# Create a directed graph
G = nx.DiGraph()

# Add edges (representing links between pages)
G.add_edges_from([(1, 2), (2, 3), (3, 1), (3, 4)])

# Compute PageRank
pagerank = nx.pagerank(G, alpha=0.85)  # alpha is the damping factor
print(pagerank)  # Dictionary with nodes as keys and PageRank scores as values


# Idea: Use the json as str to calculate the tf-idf


This is inspired by the TCC of Allan Juan based on his codebase

First needs to download the dataset

In [ ]:
folder_path_dataset = Path("/home/danielrezende/datasets/CodeforcesSourceCodeSubmissionsDataset")
folders = [f for f in folder_path_dataset.iterdir() if f.is_dir()]
folders[:10]

In [ ]:
folders[0].name

In [67]:
py_files_content = []
for folder_contest in folders[:1]:
    for folder_question in folder_contest.iterdir():
        for file_solution in folder_question.iterdir():
            if file_solution.is_file() and file_solution.suffix == ".py":
                with file_solution.open("r", encoding="utf-8") as f:
                    file_content = f.read()
                    try:
                        dict_content = json.dumps(code_to_json(file_content))
                        py_files_content.append(
                            {
                                "name": f"{folder_contest.name}/{folder_question.name}/{file_solution.name}",
                                "original_content": file_content,
                                "dict_content": dict_content,
                            }
                        )
                    except KeyError:
                        ...

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(x9_dict_contents)


similarity = cosine_similarity(tfidf_matrix)
similarity


In [ ]:
# Get feature names
feature_names = vectorizer.get_feature_names_out()

# Convert TF-IDF matrix to array
tfidf_array = tfidf_matrix.toarray()

# Display the TF-IDF scores
df = pd.DataFrame(tfidf_array, columns=feature_names)
print(df)

In [ ]:
threshold = 0.9
similar_pairs = []

for i in range(len(similarity)):
    for j in range(i + 1, len(similarity)):
        if similarity[i, j] >= threshold:
            similar_pairs.append((i, j, similarity[i, j]))

for pair in similar_pairs:
    idx1, idx2, sim_score = pair
    print(
        f"Documents {py_files_content[idx1]["name"]} and {py_files_content[idx2]["name"]} have a similarity score of {sim_score:.4f}"
    )
    print(
        f"============================CODE 1============================\n{py_files_content[idx1]["original_content"]}"
    )
    print(
        f"============================CODE 2============================\n{py_files_content[idx2]["original_content"]}"
    )
    print("\n")